In [1]:
!pip install torch transformers gradio pillow -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.8/321.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.7 MB/s eta 0:00:00


In [2]:
pip install easyocr -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.9/422.9 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.6/969.6 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.6/286.6 kB 22.0 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
API_KEY = userdata.get('DeepSeekV3')


In [ ]:
import easyocr
reader = easyocr.Reader(['en'])
results = reader.readtext('image.png')
extracted_text = " ".join([result[1] for result in results])

In [ ]:
extracted_text

'1 + 4 =5 2 + 5 = 12 3 + 6 = 21 8 + 11 = ?'

In [6]:
import gradio as gr
from PIL import Image
import easyocr
import requests
import json

# Initialize EasyOCR reader (English)
reader = easyocr.Reader(['en'])

# DeepSeek API configuration
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"

# If you're using Colab and storing your key in 'DeepSeekV3' user data:
from google.colab import userdata
API_KEY = userdata.get('DeepSeekV3')  # Replace with your actual API key if not in Colab

def solve_puzzle(image):
    """Extracts the puzzle from the image and sends it to DeepSeek for solving."""
    try:
        # 1. Save the uploaded image temporarily; EasyOCR uses file paths
        image_path = "uploaded_image.png"
        image.save(image_path)

        # 2. Extract text from the image using EasyOCR
        results = reader.readtext(image_path)
        extracted_text = " ".join([res[1] for res in results])

        # Standardize the text to avoid misinterpretation of "??" as "?"
        extracted_text = extracted_text.replace('??', '?')

        if "?" not in extracted_text:
            extracted_text += "?"

        print("Extracted Text:", extracted_text)  # Debugging output

        # 3. Refine the extracted text to standardize expressions
        refined_text = extracted_text.replace('x', '*').replace('X', '*').replace('=', ' = ').strip()
        print("Refined Text:", refined_text)  # Debugging output

        # 4. Compose the user message with detailed reasoning instructions
        puzzle_prompt = (
            f"You are an AI specialized in solving puzzles. Analyze the following, identify hidden patterns or rules, and provide the missing value with step-by-step reasoning in text format. Do not return answer in Latex."
            f"\nPuzzle:\n{refined_text}\n"
            "Format your response strictly as follows:\n"
            "1. **Given Equation**:\n   - (original equations)\n"
            "2. **Pattern Identified**:\n   (explain the hidden logic)\n"
            "3. **Step-by-step Calculation**:\n   - For (input values):\n     (calculation and result)\n"
            "4. **Final Answer**:\n     (Answer = X)"
        )

        messages = [
            {"role": "user", "content": puzzle_prompt}
        ]

        # 5. Optimized API request for structured reasoning response
        data = {
            "model": "deepseek-chat",
            "messages": messages,
            "temperature": 0,  # Reduce randomness for direct answers
            "max_tokens": 500  # Allow enough tokens for explanation
        }

        headers = {
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        }

        # 6. Send the request to DeepSeek with a timeout
        response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data, timeout=15)

        # 7. Check the result
        if response.status_code == 200:
            try:
                json_resp = response.json()
                response_content = json_resp.get("choices", [{}])[0].get("message", {}).get("content", "").strip()

                # Additional refinement if the API response needs formatting
                if response_content:
                    formatted_response = response_content.replace("Step-by-step Calculation:", "3. **Step-by-step Calculation**:").replace("Final Answer:", "4. **Final Answer**:")
                    return formatted_response
                else:
                    return "Error: No response content."
            except json.JSONDecodeError:
                return "Error: Invalid JSON response from DeepSeek API."
        else:
            return f"Error: DeepSeek API failed with status code {response.status_code}, Response: {response.text}"
    except requests.exceptions.Timeout:
        return "Error: DeepSeek API request timed out. Please try again."
    except Exception as e:
        return f"Error: {str(e)}"

# Gradio interface
interface = gr.Interface(
    fn=solve_puzzle,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Logic Puzzle Solver with EasyOCR & DeepSeek",
    description="Upload an image of a logic puzzle, and the model will solve it with step-by-step reasoning."
)

interface.launch(debug=True)


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5dba6120b899253f91.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Extracted Text: 1 + 4 =5 2 + 5 = 12 3 + 6 = 21 8 + 11 = ?
Refined Text: 1 + 4  = 5 2 + 5  =  12 3 + 6  =  21 8 + 11  =  ?
Extracted Text: 1 + 4 =5 2 + 5 = 12 3 + 6 = 21 8 + 11 = ?
Refined Text: 1 + 4  = 5 2 + 5  =  12 3 + 6  =  21 8 + 11  =  ?
Extracted Text: 2 5 3 6 1 8 2 3 7?
Refined Text: 2 5 3 6 1 8 2 3 7?
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5dba6120b899253f91.gradio.live
